# Use Case 3 — Multi-Category Classification (R + ellmer + OpenAI)

**PMRC 2026 workshop.** Classify the **author** of a Twitter bio into one of four user types, then evaluate the runs on the TaMPER floor: **Compliance · Accuracy · Precision**.

Fully in **R** using [`ellmer`](https://ellmer.tidyverse.org/) and **OpenAI** models, so it runs in Google Colab with your own OpenAI key. (`ellmer` is the same toolchain as the image-annotation lab — one helper for the LLM call, plus *structured output* that guarantees a clean, machine-readable result.)

> ⚠️ **First, set the runtime to R:** *Runtime ▸ Change runtime type ▸ R*, then run the cells in order.

## 1. Setup — connect to an LLM

Run once per Colab session. Only `ellmer` needs installing; tidyverse is pre-installed on Colab's R runtime.

In [4]:
install.packages('ellmer')

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘coro’




In [5]:
suppressMessages({
  # Try loading ellmer, but don't let it stop tidyverse from loading
  try(library(ellmer), silent = TRUE) # LLM calls + structured output
  library(tidyverse)                  # data wrangling
})

## 2. Your OpenAI key

In [ ]:
Sys.setenv(OPENAI_API_KEY = "sk-...PASTE-YOUR-KEY-HERE...")

In [ ]:
# quick connection test (should print: ready)
chat_openai(model = "gpt-4o-mini")$chat("Reply with one word: ready", echo = "none")

## 3. The task

- **Input:** 39 synthetic Twitter **bios** about a city's bid to attract a high-tech innovation center.
- **Goal:** label each author as one of four user types.
- **Reference:** each bio has a human **gold standard** label (`usertype_label`) for scoring.

**Columns in `synthetic_workshop_tweets.csv`:**

| Column | Description |
|---|---|
| `id` | Row identifier. |
| `synthetic_tweet` | The synthetic tweet text. |
| `synthetic_bio` | The author's Twitter bio — **the text we classify**. |
| `policy_context` | The policy scenario the tweet refers to. |
| `usertype_label` | Human **gold standard** user-type label — the classification target. |
| `stance_label` | Human gold standard stance label (not used in this notebook). |
| `sampled_for` | Which task the row was sampled for. |

The next cell loads the data straight from the workshop repo — no upload needed.

In [6]:
# loads the data straight from the workshop repo (no upload needed)
tweets <- read_csv(
  "https://raw.githubusercontent.com/mro0001/pmrc-genai-workshop/main/use-case-3/synthetic_workshop_tweets.csv",
  show_col_types = FALSE)
cat("bios:", nrow(tweets), "\n")
count(tweets, usertype_label)

bios: 39 


usertype_label,n
<chr>,<int>
General Public,22
Media,7
Other Stakeholders,5
Politicians/Government Accounts,5


## 4. Models & settings

Three OpenAI models and two temperatures (0 = deterministic, 1 = creative). The `PROMPT_CATEGORIES` are the exact labels the prompt asks the model to emit.

In [7]:
MODELS       <- c("gpt-4o-mini", "gpt-4.1-mini", "gpt-4o")
TEMPERATURES <- c(0, 1)

PROMPT_CATEGORIES <- c("General Public", "Business Stakeholder",
                       "Media", "Politician or Government Account")

# Keep a quick demo cheap; raise these to scale toward the full study.
DEMO_N <-  6  # bios to classify. You can select up to 39 (which is the full sample)
N_ITER <- 3   # repeated runs per bio (precision needs >= 2)

## 5. The prompts

Two prompts, identical task, different detail (verbatim from the study). `{text}` is the slot we fill with each bio.

In [14]:
REFORM_PROMPT <- r"---(
Task Overview: You are going to classify the author of a given Twitter bio into one of 4 distinct categories, General Public, Business Stakeholder, Media, Politicians and Government Accounts and provide a brief explanation for your decision.

Definitions: Business Stakeholder- Private-sector or quasi-private actors that stand to gain directly from the deal and actively promote it. This group comprises corporate representatives (e.g., Amazon executives or spokespersons), commercial real-estate developers, site-selection consultants, lobbyists, and trade-association accounts. Their messages tend to highlight job creation, tax-base growth, and the competitive advantages of their locality, often using language such as "investment," "opportunity," or "bring Amazon home."
Politician or Government Account-Actors who hold formal authority in the policy-making process. They include elected officials (mayors, governors, city council members) who can publicly claim credit or deflect blame for an economic-development deal, and career civil servants (city planners, economic-development staff, agency heads) who design, negotiate, and implement incentive packages. On Twitter they usually tweet from verified or official-government handles, reference "our city," "we're offering," or cite legislative actions.
Media-Information-dissemination entities that report on, interpret, and frame the HQ2 competition for the public. This includes local-news stations, regional newspapers, national business press, and individual reporters' personal accounts.
General Public-All remaining individual users who do not belong to the three organized groups above. They are ordinary residents, voters, community-organizers, or "concerned citizens."

Task: carefully read the author biography from a Twitter bio, identify which of the four categories the author belongs to, and provide a brief explanation for your decision.

Twitter Bio: {text}

Instructions:
2) Return the output as valid JSON in the following structure {Category: <"Business Stakeholder"|"Politician or Government Account"|"Media"|"General Public">, Explanation: <Insert Explanation Here>}.
3) Determine if the author is best categorized as "Business Stakeholder", "Politician or Government Account", "Media", or "General Public".
4) Review your choice and the corresponding definition for the Twitter Bio: {text}
5) Provide a brief explanation that justifies your choice over the other options.

Output:
{Category: <"Business Stakeholder"|"Politician or Government Account"|"Media"|"General Public">, Explanation: <Insert Explanation Here>})---"

In [15]:
ORG_PROMPT <- r"---(
I want you to perform a data annotation task. Your task is to carefully read the text and identify the category that best matches the Twitter Bio Description.
Your response must belong to one of the four categories: Business Stakeholder, Politician or Government Account, Media, or General Public.
In your output, only respond with the name of the polarity: Business Stakeholder, Politician or Government Account, Media, or General Public, depending on the information in the Twitter Bio provided. In your output, I also want you to provide an explanation for the output.

Provide your response in the first line and provide the explanation for your response in the second line.
Twitter Bio: <{text}>)---"

In [16]:
PROMPTS <- list(reform = REFORM_PROMPT, org = ORG_PROMPT)

## 6. Structured output — the `ellmer` way

Rather than *asking* for JSON in the prompt, we define a **schema**. `type_enum` forces the model to return exactly one of the four labels, and `$chat_structured()` returns a parsed object — no JSON parsing, no stray prose. (Same pattern as the image lab's `type_object` / `type_enum`.)

In [11]:
type_classification <- type_object(
  category    = type_enum(PROMPT_CATEGORIES, description = "The single best user type."),
  explanation = type_string("One short sentence justifying the choice.")
)

## 7. One classification call

`structured = TRUE` uses the schema above; `structured = FALSE` takes free text and recovers the label by matching — so we can compare the two later.

In [9]:
classify <- function(bio, model, temperature = 0, prompt = "reform", structured = TRUE) {
  msg <- gsub("{text}", bio, PROMPTS[[prompt]], fixed = TRUE)   # fill the {text} slot
  ch  <- chat_openai(model = model, params = params(temperature = temperature))
  if (structured) {
    res <- tryCatch(ch$chat_structured(msg, type = type_classification), error = function(e) NULL)
    category    <- if (is.null(res)) NA_character_ else as.character(res$category)
    explanation <- if (is.null(res)) NA_character_ else as.character(res$explanation)
  } else {
    txt <- tryCatch(as.character(ch$chat(msg, echo = "none")), error = function(e) NA_character_)
    hit <- PROMPT_CATEGORIES[vapply(PROMPT_CATEGORIES, function(c) grepl(c, txt, fixed = TRUE), logical(1))]
    category    <- if (length(hit) == 0) NA_character_ else hit[1]
    explanation <- txt
  }
  tibble(category = category, explanation = explanation,
         compliant = !is.na(category) && category %in% PROMPT_CATEGORIES)
}

## 8. Run it — repeated classifications

`N_ITER` runs of each bio for every model (the basis for precision). Start small with `DEMO_N` bios; raise `DEMO_N` / `N_ITER` to scale up.

In [17]:
set.seed(1)
# bios <- tweets %>% group_by(usertype_label) %>% slice_sample(n = 2) %>% ungroup()
bios <- tweets %>% slice_head(n = DEMO_N)
cat("Calls:", nrow(bios), "bios x", length(MODELS), "models x", N_ITER, "iters =",
    nrow(bios) * length(MODELS) * N_ITER, "\n")

runs <- tidyr::expand_grid(mi = seq_len(nrow(bios)), model = MODELS, iteration = seq_len(N_ITER)) %>%
  purrr::pmap_dfr(function(mi, model, iteration) {
    b <- bios[mi, ]
    classify(b$synthetic_bio, model, temperature = 0, prompt = "reform", structured = TRUE) %>%
      mutate(id = b$id, gold = b$usertype_label, model = model, iteration = iteration, .before = 1)
  })

runs %>% select(id, model, iteration, category, gold) %>% head(8)

tibble [8 × 7] (S3: tbl_df/tbl/data.frame)
 $ id             : num [1:8] 4 23 11 12 10 8 18 17
 $ synthetic_tweet: chr [1:8] "#TechHubMove \nTechCorp Was Never Going to Deliver 20,000 Jobs in Riverdale https://t.co/xWzQrTgHjK via @CityChronicle" "#TechHQMove great insights here......support local growth where it’s needed most!! https://t.co/XFVZaQ8dKl" "#TechHQ2虚构市 This article gets to the heart of a key factor in the HQ2 race: The Digital Times reports in @Te"| __truncated__ "#TechCorpHQ2 #TechCorp #BayviewCity #California #RealEstate https://t.co/F8jKl2Zx9Q" ...
 $ synthetic_bio  : chr [1:8] "standing up for what's right, finding joy in the little things, dreaming of a more peaceful future." "exploring the cosmos for stellar tunes and tacos since 1992. future pet enthusiast. she/her with a sprinkle of "| __truncated__ "senior reporter,虚构市商业周刊, real estate and related industries beat. other interests: news, finance, photog"| __truncated__ "media consultant / communications expert. cra

id,model,iteration,category,gold
<dbl>,<chr>,<int>,<chr>,<chr>
4,gpt-4o-mini,1,NA,General Public
4,gpt-4o-mini,2,NA,General Public
4,gpt-4o-mini,3,NA,General Public
4,gpt-4.1-mini,1,NA,General Public
4,gpt-4.1-mini,2,NA,General Public
4,gpt-4.1-mini,3,NA,General Public
4,gpt-4o,1,NA,General Public
4,gpt-4o,2,NA,General Public


## 9. Compliance — did it emit valid labels?

The operational gate: did the output use one of the predefined labels (and nothing missing)?

In [ ]:
runs %>% group_by(model) %>%
  summarise(`valid label %` = round(mean(category %in% PROMPT_CATEGORIES) * 100, 1),
            `missing %`     = round(mean(is.na(category)) * 100, 1), .groups = "drop")

## 10. Accuracy — vs the human gold standard

The prompt and the gold standard use slightly different wording (`Business Stakeholder` vs `Other Stakeholders`), so we map both onto one scheme with `canon()`, then take the exact match.

In [ ]:
canon <- function(x) {
  s <- tolower(trimws(gsub("[.,;:]+$", "", trimws(as.character(x)))))
  dplyr::case_when(
    grepl("stakeholder", s)                                ~ "Other Stakeholders",
    grepl("^politician", s) | grepl("government", s)       ~ "Politicians/Government Accounts",
    s %in% c("media", "news", "news media")                ~ "Media",
    grepl("public", s)                                     ~ "General Public",
    TRUE                                                   ~ NA_character_)
}

scored <- runs %>% mutate(pred = canon(category), truth = canon(gold))
scored %>% filter(!is.na(pred), !is.na(truth)) %>%
  group_by(model) %>% summarise(`accuracy %` = round(mean(pred == truth) * 100, 1), .groups = "drop")

## 11. McNemar — are the model gaps real?

A paired test: do two models differ on the *same* bios, or is the accuracy gap noise? We use the exact (binomial) form on the discordant cells.

In [ ]:
mcnemar_pair <- function(scored, m1, m2) {
  w <- scored %>% filter(!is.na(pred), !is.na(truth), model %in% c(m1, m2)) %>%
    mutate(correct = as.integer(pred == truth)) %>%
    select(id, iteration, model, correct) %>%
    tidyr::pivot_wider(names_from = model, values_from = correct) %>%
    tidyr::drop_na(all_of(c(m1, m2)))
  b  <- sum(w[[m1]] == 1 & w[[m2]] == 0)     # m1 right, m2 wrong
  cc <- sum(w[[m1]] == 0 & w[[m2]] == 1)     # m2 right, m1 wrong
  tibble(comparison = paste(m1, "vs", m2), `m1>m2` = b, `m2>m1` = cc,
         `McNemar p` = if (b + cc == 0) 1 else round(binom.test(b, b + cc, 0.5)$p.value, 3))
}

purrr::map_dfr(combn(MODELS, 2, simplify = FALSE), ~ mcnemar_pair(scored, .x[1], .x[2]))

## 12. Precision — same answer across runs?

Consistency of the label across the `N_ITER` repeats: agreement rate (all runs identical) plus Krippendorff's nominal alpha.

In [ ]:
kripp_alpha_nominal <- function(m) {
  cats <- sort(unique(as.vector(m[!is.na(m)])))
  if (length(cats) == 0) return(NA_real_)
  O <- matrix(0, length(cats), length(cats), dimnames = list(cats, cats))
  for (u in seq_len(nrow(m))) {
    v <- m[u, ]; v <- v[!is.na(v)]; mu <- length(v)
    if (mu < 2) next
    for (a in seq_len(mu)) for (bb in seq_len(mu)) if (a != bb)
      O[v[a], v[bb]] <- O[v[a], v[bb]] + 1 / (mu - 1)
  }
  n <- sum(O); nc <- rowSums(O)
  den <- sum(nc)^2 - sum(nc^2)
  if (den == 0) return(NA_real_)
  1 - (n - 1) * (n - sum(diag(O))) / den
}

runs %>% mutate(pred = canon(category)) %>% filter(!is.na(pred)) %>%
  group_by(model) %>% group_modify(function(g, ...) {
    w   <- g %>% distinct(id, iteration, pred) %>%
      tidyr::pivot_wider(names_from = iteration, values_from = pred)
    mat <- as.matrix(w %>% select(-id))
    agree <- mean(apply(mat, 1, function(v) { v <- v[!is.na(v)]; length(v) >= 2 && length(unique(v)) == 1 }))
    tibble(`agreement %` = round(agree * 100, 1),
           `Krippendorff alpha` = round(kripp_alpha_nominal(mat), 3))
  }) %>% ungroup()

## 13. Compare the design choices

Hold the **model** fixed and change one thing at a time — prompt, temperature, then structured output — reporting all three criteria (compliance, accuracy, precision) plus a paired **McNemar** accuracy test. The reference arm is **reform / temp 0 / structured** (one model, to keep it cheap).

In [ ]:
# Helpers: run one condition, summarise the three criteria, and pair two arms with McNemar.
run_condition <- function(bios, model, temperature, prompt, structured, n_iter = N_ITER) {
  tidyr::expand_grid(mi = seq_len(nrow(bios)), iteration = seq_len(n_iter)) %>%
    purrr::pmap_dfr(function(mi, iteration) {
      b <- bios[mi, ]
      classify(b$synthetic_bio, model, temperature, prompt, structured) %>%
        mutate(id = b$id, gold = b$usertype_label, iteration = iteration, .before = 1)
    })
}

metrics_one <- function(rn) {                              # compliance, accuracy, precision for one arm
  pred <- canon(rn$category); truth <- canon(rn$gold); ok <- !is.na(pred) & !is.na(truth)
  w   <- tibble(id = rn$id, iteration = rn$iteration, pred = pred) %>% filter(!is.na(pred)) %>%
    distinct(id, iteration, pred) %>% tidyr::pivot_wider(names_from = iteration, values_from = pred)
  mat <- as.matrix(w %>% select(-id))
  agree <- mean(apply(mat, 1, function(v) { v <- v[!is.na(v)]; length(v) >= 2 && length(unique(v)) == 1 }))
  tibble(`compliance %` = round(mean(rn$category %in% PROMPT_CATEGORIES) * 100, 1),
         `accuracy %`   = round(mean(pred[ok] == truth[ok]) * 100, 1),
         `agreement %`  = round(agree * 100, 1),
         alpha          = round(kripp_alpha_nominal(mat), 3))
}

mcnemar_arms <- function(rnA, rnB) {                       # paired exact McNemar on accuracy
  jn <- inner_join(
    tibble(id = rnA$id, iteration = rnA$iteration, a = as.integer(canon(rnA$category) == canon(rnA$gold))),
    tibble(id = rnB$id, iteration = rnB$iteration, b = as.integer(canon(rnB$category) == canon(rnB$gold))),
    by = c("id", "iteration")) %>% tidyr::drop_na(a, b)
  bb <- sum(jn$a == 1 & jn$b == 0); cc <- sum(jn$a == 0 & jn$b == 1)
  if (bb + cc == 0) 1 else round(binom.test(bb, bb + cc, 0.5)$p.value, 3)
}

compare_arms <- function(rnA, rnB, labA, labB) {
  print(bind_rows(metrics_one(rnA) %>% mutate(arm = labA, .before = 1),
                  metrics_one(rnB) %>% mutate(arm = labB, .before = 1)))
  cat("McNemar (accuracy)  ", labA, "vs", labB, ":  p =", mcnemar_arms(rnA, rnB), "\n")
}

cmp_model <- MODELS[1]                                     # one model for the comparisons
# cmp_bios  <- bios
cmp_bios <- tweets %>% slice_head(n = DEMO_N)
ref_run   <- runs %>% filter(model == cmp_model)          # reform / temp 0 / structured (from section 8)

### Prompt — reform vs org  *(temp 0, structured)*

In [ ]:
org_run <- run_condition(cmp_bios, cmp_model, temperature = 0, prompt = "org", structured = TRUE)
compare_arms(ref_run, org_run, "reform", "org")

### Temperature — 0 vs 1  *(reform, structured)*

In [ ]:
t1_run <- run_condition(cmp_bios, cmp_model, temperature = 1, prompt = "reform", structured = TRUE)
compare_arms(ref_run, t1_run, "temp 0", "temp 1")

### Structured output — schema vs free text  *(reform, temp 0)*

In [ ]:
plain_run <- run_condition(cmp_bios, cmp_model, temperature = 0, prompt = "reform", structured = FALSE)
compare_arms(ref_run, plain_run, "structured", "unstructured")

**Reading the comparisons.** Prompt usually moves accuracy the most; temperature mainly trades **precision**; structured output's payoff is largest with a weak prompt. The three criteria say different things: **compliance** = usable format, **accuracy** = closeness to the human gold standard, **precision** = consistency across runs.